In [5]:
from carcara.sampler.random_displacements import RandomDisplacements
from ase.build import bulk
from ase import Atoms
from ase.io import write, read
from ase.calculators.emt import EMT
from pathlib import Path

In [2]:
atoms = bulk("Au", "fcc", a=4.08, cubic=True).repeat((3,3,3))
calculator = EMT()

In [ ]:
generator = RandomDisplacements(atoms=atoms, calculator=calculator, seed=42)
generator.relax_structure(fmax=0.01, relax_cell=True)
generator.generate_samples(num_samples=500,
                           noise_type='normal',
                           noise_level_pos=0.2,
                           noise_level_cell=0.2,
                           cell_mode='all',
                           compute_energy_and_forces=True)
output_dir = Path("data/raw")
output_dir.mkdir(parents=True, exist_ok=True)
generator.write_xyz(output_dir / "noise_samples.xyz")

Dataset saved successfully in data/raw/noise_samples.xyz


In [ ]:
# Checking the generated samples
generator.summary()

Generated 500 samples.
Cell deviation: mean = 0.5784, std = 0.1410
Position deviation: mean = 4.7474, std = 0.6412
Energy: mean = 58.8226 eV, std = 13.0005 eV
Forces: mean = 0.0000 eV/Å, std = 3.9389 eV/Å


In [ ]:
# isolated atom (Au)
isolated_atom = Atoms("Au", positions=[[0, 0, 0]], cell=[10, 10, 10])
chemical_symbols = isolated_atom.get_chemical_symbols()
isolated_atom.calc = EMT()
energy = isolated_atom.get_potential_energy()
print(f"Isolated atom energy: {energy:.4f} eV")
output_dir = Path("data/raw")
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)

isolated_atom.info['REF_energy'] = energy
isolated_atom.calc = None
write(output_dir / f"{chemical_symbols[0]}_atom.xyz", isolated_atom)

Isolated atom energy: 3.8000 eV
